# LangChain 智能旅游助手 / LangChain Travel Assistant

本笔记本演示了基于 LangChain 和智谱AI (GLM-4) 的智能旅游助手的完整实现。

This notebook demonstrates a complete implementation of an intelligent travel assistant built with LangChain and ZhipuAI (GLM-4).

## 1. 安装依赖 / Install Dependencies

In [ ]:
!pip install langchain==0.3.13 langchain-community pyjwt requests python-dotenv

## 2. 配置 / Configuration

In [ ]:
import os

# Set your API key here or use .env file
# os.environ['ZHIPUAI_API_KEY'] = 'your_api_key_here'

ZHIPUAI_API_KEY = os.getenv('ZHIPUAI_API_KEY')
MODEL_NAME = 'glm-4'
TEMPERATURE = 0.5

print(f'API Key configured: {bool(ZHIPUAI_API_KEY)}')
print(f'Model: {MODEL_NAME}')
print(f'Temperature: {TEMPERATURE}')

## 3. 定义工具 / Define Tools

In [ ]:
import requests
from langchain.agents import Tool

def query_weather(city: str) -> str:
    """Query weather information for a city"""
    api_url = f'http://wttr.in/{city}?format=3'
    try:
        response = requests.get(api_url, timeout=10)
        return response.text
    except Exception as e:
        return f'Unable to get weather for {city}: {str(e)}'

weather_tool = Tool(
    name='Weather Query',
    func=query_weather,
    description='Query current weather information for a city. Input city name.',
)

print('Weather tool test:')
print(query_weather('Beijing'))

In [ ]:
def calculate_budget_internal(daily_budget: float, days: int) -> str:
    """Calculate total travel budget based on daily budget and number of days"""
    try:
        total_budget = daily_budget * days
        return f'Your travel budget is {total_budget:.2f} yuan.'
    except Exception as e:
        return f'Unable to calculate budget: {str(e)}'

def calculate_budget(input_data: str) -> str:
    """Parse input and calculate budget"""
    try:
        parts = input_data.split(',')
        if len(parts) != 2:
            return "Input format error. Please use format: 'daily_budget,days' e.g., '500,3'"
        daily_budget = float(parts[0].strip())
        days = int(parts[1].strip())
        return calculate_budget_internal(daily_budget, days)
    except ValueError:
        return "Input format error. Please use format: 'daily_budget,days' e.g., '500,3'"

budget_tool = Tool(
    name='Budget Calculator',
    func=calculate_budget,
    description="Calculate total travel budget based on daily budget and days. Input format: 'daily_budget,days', e.g., '500,3'.",
)

print('Budget tool test:')
print(calculate_budget('700,3'))

In [ ]:
from langchain_community.chat_models import ChatZhipuAI
from langchain.schema import SystemMessage, HumanMessage

chat = ChatZhipuAI(
    api_key=ZHIPUAI_API_KEY,
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

def recommend_place(question: str) -> str:
    """Use ZhipuAI to answer travel-related questions"""
    messages = [
        SystemMessage(content='Your role is a travel assistant.'),
        HumanMessage(content=question),
    ]
    response = chat.invoke(messages)
    return response.content

place_tool = Tool(
    name='Travel Recommendation',
    func=recommend_place,
    description='Answer user questions about travel, such as recommending attractions. Input question description.',
)

## 4. 初始化 Agent / Initialize Agent

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

tools = [weather_tool, budget_tool, place_tool]
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

agent = initialize_agent(
    tools=tools,
    llm=chat,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
)

print('Agent initialized successfully!')

## 5. 测试对话 / Test Conversations

In [ ]:
# Test 1: General travel advice
response = agent.run('请介绍一下北京的旅游情况')
print('Test 1 Response:', response)

In [ ]:
# Test 2: Weather query
response = agent.run('北京近期的天气如何？')
print('Test 2 Response:', response)

In [ ]:
# Test 3: Budget calculation
response = agent.run('我打算在北京玩3天，每天的预算是700元，请帮我计算总开销？')
print('Test 3 Response:', response)

In [ ]:
# Test 4: Attraction recommendations
response = agent.run('请推荐北京的景点')
print('Test 4 Response:', response)